# Milestone 2 RAG Exploration

This notebook explores the Milestone 2 backend workflow for the Amazon Reviews 2023 **Video_Games** project.

It includes:

- LLM/API setup and sanity checks
- semantic retrieval + semantic RAG
- hybrid retrieval + hybrid RAG
- prompt variant comparison
- backend tests that can replace terminal testing
- saved example outputs for later discussion and UI handoff

This notebook assumes:
- raw data and processed Milestone 1 sample artifacts already exist
- a local `.env` file exists in the project root
- the `.env` file contains:
  - `GROQ_API_KEY`
  - `LLM_MODEL` (for example `llama-3.1-8b-instant`)


## Workflow diagram

```mermaid
flowchart TD
    A[User Query] --> B[Retriever]
    B --> C[Top-k Documents]
    C --> D[Context Builder]
    D --> E[Prompt Template]
    E --> F[Hosted LLM API]
    F --> G[Generated Answer]

    subgraph Semantic RAG
        B1[Semantic Retriever / FAISS]
    end

    subgraph Hybrid RAG
        B2[BM25 Retriever]
        B3[Semantic Retriever]
        B4[Hybrid Merger / RRF]
    end

    A --> B1
    B1 --> C

    A --> B2
    A --> B3
    B2 --> B4
    B3 --> B4
    B4 --> C
```


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

from langchain_groq import ChatGroq

from src.semantic import SemanticRetriever
from src.bm25 import BM25Retriever
from src.hybrid import HybridRetriever
from src.rag_pipeline import SemanticRAGPipeline, HybridRAGPipeline

/Users/lukeni/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths and configuration

In [2]:
processed_dir = PROJECT_ROOT / "data" / "processed"

CORPUS_PATH = processed_dir / "video_games_corpus_sample.csv"
BM25_INDEX_PATH = processed_dir / "bm25_sample_index.pkl"
BM25_TOKENS_PATH = processed_dir / "bm25_sample_tokens.pkl"
FAISS_INDEX_PATH = processed_dir / "faiss_sample.index"
SEMANTIC_METADATA_PATH = processed_dir / "semantic_sample_metadata.pkl"

LLM_MODEL = os.getenv("LLM_MODEL", "llama-3.1-8b-instant")

print("Processed dir:", processed_dir)
print("LLM model:", LLM_MODEL)

Processed dir: /Users/lukeni/Desktop/ubc/MDS/TERM6/575/DSCI_575_project_li0606_Lukeni/data/processed
LLM model: llama-3.1-8b-instant


## Check that Milestone 1 artifacts exist

In [3]:
required_files = [
    CORPUS_PATH,
    BM25_INDEX_PATH,
    BM25_TOKENS_PATH,
    FAISS_INDEX_PATH,
    SEMANTIC_METADATA_PATH,
]

for path in required_files:
    print(path.name, "->", path.exists())

video_games_corpus_sample.csv -> True
bm25_sample_index.pkl -> True
bm25_sample_tokens.pkl -> True
faiss_sample.index -> True
semantic_sample_metadata.pkl -> True


## API key sanity check

In [4]:
print("Has GROQ_API_KEY:", bool(os.getenv("GROQ_API_KEY")))
print("Model from env:", os.getenv("LLM_MODEL"))

Has GROQ_API_KEY: True
Model from env: llama-3.1-8b-instant


## Test the Groq API directly

In [5]:
llm = ChatGroq(
    model=LLM_MODEL,
    api_key=os.getenv("GROQ_API_KEY"),
)

response = llm.invoke("Say hello in one short sentence.")
print(response.content)

Hello.


## Model choice and rationale

For Milestone 2, we use a hosted LLM through the Groq API instead of running a local model.
This is useful because:

- it avoids local model downloads and hardware constraints
- it keeps setup lighter for a laptop-based project
- it lets us focus on retrieval, prompts, and grounded generation

The current model is controlled by the `.env` variable `LLM_MODEL`.


## Test the semantic retriever only

In [6]:
semantic_retriever = SemanticRetriever.load_saved(
    index_path=FAISS_INDEX_PATH,
    metadata_path=SEMANTIC_METADATA_PATH,
)

semantic_docs = semantic_retriever.search(
    "story-rich scary game with dark atmosphere",
    top_k=5,
)

semantic_docs[["product_title", "review_title", "rating", "semantic_score"]].head()

/Users/lukeni/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


,product_title,review_title,rating,semantic_score
0,Among the Sleep: Enhanced Edition - Nintendo S...,Hated this game.,1.0,0.557222
1,Scratches: Directors Cut - PC,A haunting vacation you'll never forget...,4.0,0.492031
2,Bad Mojo: Redux - PC,"A gritty, edgy homage to Kafka and Lynch that ...",5.0,0.487693
3,Broken Sword: Sleeping Dragon - PC,George and Nico are back in style!,5.0,0.485437
4,Silent Hill: Downpour - Xbox 360,Never finished it,3.0,0.469734


## Semantic RAG pipeline

In [7]:
semantic_rag = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
)

/Users/lukeni/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### Test semantic RAG with one query

In [8]:
semantic_result = semantic_rag.answer("story-rich scary game with dark atmosphere")

print("=== SEMANTIC RAG ANSWER ===")
print(semantic_result["answer"])

=== SEMANTIC RAG ANSWER ===
Based on the provided context, I would recommend the following games:

1. Scratches: Directors Cut - PC (ASIN: B000KMCF0G) - This game is described as a horror game with a dark atmosphere, and the reviewer mentions that it's "dark, dark motivations, dark secrets await you." The game is set in a dilapidated Victorian mansion, and the reviewer praises its ability to create a foreboding and evil atmosphere.

2. Bad Mojo: Redux - PC (ASIN: B0006AAOJQ) - This game is described as a gritty, edgy horror game with a dark atmosphere. The reviewer mentions that the game contains numerous gross-outs and is not for the squeamish, but it adds to the atmosphere of the dark, seedy world of Eddie's Bar.

3. Broken Sword: Sleeping Dragon - PC (ASIN: B0000A3442) - While not strictly a scary game, Broken Sword: Sleeping Dragon has a dark and suspenseful atmosphere, especially in its 3D prerendered platform. The game has a variety of puzzle types, most of moderate difficulty, w

In [9]:
semantic_result["docs"][["product_title", "review_title", "rating"]].head()

,product_title,review_title,rating
0,Among the Sleep: Enhanced Edition - Nintendo S...,Hated this game.,1.0
1,Scratches: Directors Cut - PC,A haunting vacation you'll never forget...,4.0
2,Bad Mojo: Redux - PC,"A gritty, edgy homage to Kafka and Lynch that ...",5.0
3,Broken Sword: Sleeping Dragon - PC,George and Nico are back in style!,5.0
4,Silent Hill: Downpour - Xbox 360,Never finished it,3.0


In [10]:
print("=== CONTEXT PREVIEW ===")
print(semantic_result["context"][:1500])

=== CONTEXT PREVIEW ===
ASIN: B07NDZZHPF
Title: Among the Sleep: Enhanced Edition - Nintendo Switch
Rating: 1.0
Review title: Hated this game.
Review text: Bad graphics and questionable storyline. You are a baby escaping an abusive mother. It was just weird and not fun.


ASIN: B000KMCF0G
Title: Scratches: Directors Cut - PC
Rating: 4.0
Review title: A haunting vacation you'll never forget...
Review text: In the independently developed Scratches, you play as Michael Arthate, a British horror writer who's hard-pressed to finish his sophomore novel. In an attempt to seek inspiration, you arrange to stay at a dilapidated Victorian manner in the English countryside. The next three days will change your life forever. If I had to sum up Scratches in one word, it would be dark. Dark ambiance, dark motivations, dark secrets await you.<br /><br />The game is set in the year 1976, and during the course of your investigations you'll revisit the shocking past of Blackwood Manor. Feverish dreams (o

## Prompt variants

In [11]:
SYSTEM_PROMPT_V1 = """
You are a helpful Amazon shopping assistant.
Answer the question using ONLY the provided context from Amazon product reviews and metadata.
Do not make up product details that are not in the context.
When possible, mention the product title or ASIN.
"""

SYSTEM_PROMPT_V2 = """
You are a careful product recommendation assistant.
Use only the retrieved Amazon review context below.
If the context is insufficient, say so clearly.
Keep the answer concise, helpful, and grounded in the retrieved evidence.
"""

SYSTEM_PROMPT_V3 = """
You are an Amazon reviews analyst.
Answer the user only from the retrieved reviews and product metadata.
Prefer direct evidence from the retrieved context, and avoid unsupported claims.
"""

In [12]:
semantic_rag_v1 = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V1,
)

semantic_rag_v2 = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V2,
)

semantic_rag_v3 = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V3,
)

/Users/lukeni/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/lukeni/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/lukeni/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [13]:
test_query = "best racing game with fun tracks"

ans_v1 = semantic_rag_v1.answer(test_query)["answer"]
ans_v2 = semantic_rag_v2.answer(test_query)["answer"]
ans_v3 = semantic_rag_v3.answer(test_query)["answer"]

print("=== Prompt V1 ===")
print(ans_v1)
print("\n=== Prompt V2 ===")
print(ans_v2)
print("\n=== Prompt V3 ===")
print(ans_v3)

=== Prompt V1 ===
Based on the provided context, I would recommend the "Cars 3: Driven to Win - Xbox 360" (ASIN: B06Y6GPGQ8). 

The review for this game mentions that there are "tracks with fun shortcuts to explore" and "practice maps to enjoy for fun (not timed) where you can learn to do stunts, drive, and shoot the blasters."

=== Prompt V2 ===
Based on the context, I recommend "Cars 3: Driven to Win - Xbox 360" (ASIN: B06Y6GPGQ8) for a racing game with fun tracks. The review states that the game has "a good variety of characters, tracks, and types of activities" and that "the tracks have fun shortcuts to explore too." This suggests that the game has a diverse and engaging track selection, making it a good choice for those looking for a fun racing experience.

=== Prompt V3 ===
Based on the provided reviews, the best racing game with fun tracks is likely "Cars 3: Driven to Win - Xbox 360" (ASIN: B06Y6GPGQ8). 

Review text from this game mentions that "The tracks have fun shortcuts to

### Prompt observation notes

Add short notes here after comparing the outputs:

- Which prompt gave the most grounded answer?
- Which prompt was the most concise?
- Which prompt avoided unsupported claims best?


## Test the hybrid retriever only

In [14]:
bm25_retriever = BM25Retriever.load_saved(
    corpus_path=CORPUS_PATH,
    index_path=BM25_INDEX_PATH,
    tokens_path=BM25_TOKENS_PATH,
)

semantic_retriever = SemanticRetriever.load_saved(
    index_path=FAISS_INDEX_PATH,
    metadata_path=SEMANTIC_METADATA_PATH,
)

hybrid_retriever = HybridRetriever(
    bm25_retriever=bm25_retriever,
    semantic_retriever=semantic_retriever,
    top_k=5,
)

hybrid_docs = hybrid_retriever.search(
    "story-rich scary game with dark atmosphere",
    top_k=5,
)

hybrid_docs[["product_title", "review_title", "rating", "hybrid_score"]].head()

/Users/lukeni/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


,product_title,review_title,rating,hybrid_score
0,Fatal Frame 2: Crimson Butterfly,Good luck finishing this yourself,5.0,0.016393
1,Among the Sleep: Enhanced Edition - Nintendo S...,Hated this game.,1.0,0.016393
2,Little Nightmares - Xbox One [Digital Code],Reminds me of Dark Eye from the PC.,4.0,0.016129
3,Scratches: Directors Cut - PC,A haunting vacation you'll never forget...,4.0,0.016129
4,Among the Sleep - PlayStation 4,Inserting Game!,5.0,0.015873


## Hybrid RAG pipeline

In [15]:
hybrid_rag = HybridRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    bm25_index_path=str(BM25_INDEX_PATH),
    bm25_tokens_path=str(BM25_TOKENS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
)

/Users/lukeni/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [16]:
hybrid_result = hybrid_rag.answer("story-rich scary game with dark atmosphere")

print("=== HYBRID RAG ANSWER ===")
print(hybrid_result["answer"])

=== HYBRID RAG ANSWER ===
Based on the provided context, I would recommend the following games that match your request:

1. Fatal Frame 2: Crimson Butterfly (ASIN: B0000AI1KK) - This game is described as one of the best survival horror games out there with a great atmosphere and story.
2. Scratches: Directors Cut (ASIN: B000KMCF0G) - This game is described as a dark and foreboding atmosphere with a great storytelling and setting.
3. Little Nightmares (ASIN: B01NAUEFNR) - This game is described as scary and has a dark atmosphere, although it is more focused on trial and error gameplay.
4. Among the Sleep - PlayStation 4 (ASIN: B01IPARS7Y) - This game is described as a good indie game with a dark atmosphere, but there is limited information about its story.

Note that Among the Sleep (ASIN: B07NDZZHPF) is not recommended as it has a low rating and is described as having bad graphics and a questionable storyline.


In [17]:
hybrid_result["docs"][["product_title", "review_title", "rating"]].head()

,product_title,review_title,rating
0,Fatal Frame 2: Crimson Butterfly,Good luck finishing this yourself,5.0
1,Among the Sleep: Enhanced Edition - Nintendo S...,Hated this game.,1.0
2,Little Nightmares - Xbox One [Digital Code],Reminds me of Dark Eye from the PC.,4.0
3,Scratches: Directors Cut - PC,A haunting vacation you'll never forget...,4.0
4,Among the Sleep - PlayStation 4,Inserting Game!,5.0


## Compare semantic RAG vs hybrid RAG

In [19]:
comparison_query = "story-rich scary game with dark atmosphere"

if "semantic_result" not in globals():
    semantic_result = semantic_rag.answer(comparison_query)

if "hybrid_result" not in globals():
    hybrid_result = hybrid_rag.answer(comparison_query)

print("=== SEMANTIC RAG ===")
print(semantic_result["answer"])

print("\n=== HYBRID RAG ===")
print(hybrid_result["answer"])

=== SEMANTIC RAG ===
Based on the provided context, I would recommend the following games:

1. Scratches: Directors Cut - PC (ASIN: B000KMCF0G) - This game is described as a horror game with a dark atmosphere, and the reviewer mentions that it's "dark, dark motivations, dark secrets await you." The game is set in a dilapidated Victorian mansion, and the reviewer praises its ability to create a foreboding and evil atmosphere.

2. Bad Mojo: Redux - PC (ASIN: B0006AAOJQ) - This game is described as a gritty, edgy horror game with a dark atmosphere. The reviewer mentions that the game contains numerous gross-outs and is not for the squeamish, but it adds to the atmosphere of the dark, seedy world of Eddie's Bar.

3. Broken Sword: Sleeping Dragon - PC (ASIN: B0000A3442) - While not strictly a scary game, Broken Sword: Sleeping Dragon has a dark and suspenseful atmosphere, especially in its 3D prerendered platform. The game has a variety of puzzle types, most of moderate difficulty, which ca

### Comparison notes

Add short notes here after testing:

- Did hybrid retrieval produce more relevant supporting documents?
- Did the hybrid answer look more complete?
- Was one answer more grounded than the other?


## Save example outputs

In [20]:
semantic_result["docs"].to_csv(processed_dir / "milestone2_semantic_docs_preview.csv", index=False)
hybrid_result["docs"].to_csv(processed_dir / "milestone2_hybrid_docs_preview.csv", index=False)

with open(processed_dir / "milestone2_semantic_answer.txt", "w", encoding="utf-8") as f:
    f.write(semantic_result["answer"])

with open(processed_dir / "milestone2_hybrid_answer.txt", "w", encoding="utf-8") as f:
    f.write(hybrid_result["answer"])

print("Saved example outputs to data/processed/")

Saved example outputs to data/processed/
